# Restaurant RAG — step by step

Run **one cell at a time**. Each step depends on the cells above it.

Need `OPENAI_API_KEY` in `.env`.

## 1. Imports and API key

In [ ]:
import os
from collections import Counter
from pathlib import Path

from dotenv import load_dotenv
from docx import Document as WordDocument
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

load_dotenv()

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent

print("OPENAI_API_KEY loaded:", bool(os.getenv("OPENAI_API_KEY")))
print("Project root:", PROJECT_ROOT)

## 2. Load the Word menu

`word_doc` is python-docx. It is **not** a LangChain document yet.

In [ ]:
MENU_PATH = PROJECT_ROOT / "data" / "menu.docx"

word_doc = WordDocument(MENU_PATH)
lines = [p.text.strip() for p in word_doc.paragraphs if p.text.strip()]

print("File:", MENU_PATH)
print("Paragraphs:", len(lines))
print("First 8 lines:")
for line in lines[:8]:
    print(" -", line)

## 3. Chunk: one LangChain Document per menu item

In [ ]:
CATEGORIES = [
    "Simply Pizza Range",
    "Classic Pizza Range",
    "Supreme Pizza Range",
    "Premium Pizza Range",
    "Chef's Special Pizza Range",
    "Vito's Special - XL Size Pizza",
    "Extra Toppings for Pizza",
    "Calzone",
    "Salad",
    "Pasta",
    "Lasanga",
    "Dessert",
    "Drinks",
]
SKIP_PREFIXES = ("Size:", "Choice of", "Folded", "Vito's Restaurant Menu")

docs = []
category = None
range_price = None
notes = []
i = 0

while i < len(lines):
    line = lines[i]
    if line in CATEGORIES:
        category, range_price, notes = line, None, []
        i += 1
        continue
    if line.startswith("Range Price:"):
        range_price = line.replace("Range Price:", "").strip()
        i += 1
        continue
    if line.startswith(SKIP_PREFIXES):
        notes.append(line)
        i += 1
        continue

    name = line
    price, ingredients = range_price, None
    i += 1
    while i < len(lines) and lines[i] not in CATEGORIES:
        if lines[i].startswith("Price:"):
            price = lines[i].replace("Price:", "").strip()
            i += 1
        elif lines[i].startswith("Ingredients/Description:"):
            ingredients = lines[i].replace("Ingredients/Description:", "").strip()
            i += 1
        else:
            break

    parts = [f"Category: {category}", f"Item: {name}"]
    if price:
        parts.append(f"Price: {price}")
    if ingredients:
        parts.append(f"Ingredients: {ingredients}")
    if notes:
        parts.append("Notes: " + " ".join(notes))

    docs.append(
        Document(
            page_content="\n".join(parts),
            metadata={"category": category, "item": name},
        )
    )

print(len(docs), "chunks")

## 4. Inspect chunks

Expect about 82 items, 17 drinks, 4 desserts. If a heading became an item, fix `CATEGORIES` / `SKIP_PREFIXES` and rerun step 3.

In [ ]:
print(Counter(d.metadata["category"] for d in docs))
print("\n--- first pizza ---")
print(docs[0].page_content)
print("\n--- first drink ---")
print(next(d.page_content for d in docs if d.metadata["category"] == "Drinks"))

## 5. OpenAI embeddings

This calls OpenAI (`text-embedding-3-small`). It is not the chat LLM.

In [ ]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
sample_vector = embeddings.embed_query("mojito")
print("Embedding length:", len(sample_vector))

## 6. Vector store

In [ ]:
vectorstore = FAISS.from_documents(docs, embeddings)
print("Indexed", len(docs), "chunks")

## 7. Retriever only (no LLM yet)

Confirm drinks do not pull whole dessert/topping sections.

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
retrieved = retriever.invoke("what drinks do you have")

for i, doc in enumerate(retrieved, start=1):
    print(f"{i}. [{doc.metadata['category']}] {doc.metadata['item']}")

## 8. LLM — ChatOpenAI

This is the model that writes the answer. Test it **without** the menu first.

In [ ]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    max_tokens=120,
)

raw = llm.invoke("Say hello in one short sentence.")
print(raw.content)

## 9. Prompt template

Loaded from `prompts/system.txt` and `prompts/human.txt`.
`{context}` = retrieved menu items. `{question}` = user question.

In [ ]:
PROMPT_DIR = PROJECT_ROOT / "prompts"

system_text = (PROMPT_DIR / "system.txt").read_text(encoding="utf-8")
human_text = (PROMPT_DIR / "human.txt").read_text(encoding="utf-8")

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_text),
        ("human", human_text),
    ]
)

print(system_text)
print("---")
print(prompt)

## 10. Turn retrieved docs into one context string

In [ ]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

context_preview = format_docs(retrieved)
print(context_preview[:800])

## 11. Build the chain

```
question → retriever → format_docs → prompt → llm → string
```

In [ ]:
chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)

print(type(chain))

## 12. Run the chain

In [ ]:
print(chain.invoke("What drinks do you have?"))

In [ ]:
print(chain.invoke("Do you have a vegetarian pizza?"))

In [ ]:
print(chain.invoke("Hi"))